In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
col = [0,0,1,2,1]
row = [0,1,0,2,2]
data = [1,2,3,4,5]
sparseMatrix = csr_matrix((data, (row,col)), shape = (3,3))
print(sparseMatrix)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 5 stored elements and shape (3, 3)>
  Coords	Values
  (0, 0)	1
  (0, 1)	3
  (1, 0)	2
  (2, 1)	5
  (2, 2)	4


In [2]:
sarray = sparseMatrix.toarray()
sarray

array([[1, 3, 0],
       [2, 0, 0],
       [0, 5, 4]])

In [3]:
sm = csr_matrix(sarray)
sm

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 5 stored elements and shape (3, 3)>

In [4]:
df  = pd.read_csv("movie_ratings_grid.csv", index_col = 0)
df.head()


,The_Dark_Knight,Inception,Interstellar,The_Matrix,Avengers_Endgame,Parasite,Joker,1917,Ford_v_Ferrari,Once_Upon_Hollywood,...,Barbie,Killers_Flower_Moon,Past_Lives,Saltburn,Poor_Things,Zone_of_Interest,Maestro,Priscilla,Napoleon,May_December
User_001,0,0,0,0,0,0,0,0,5,3,...,0,0,0,4,4,0,0,2,0,4
User_002,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,5,0,0,5
User_003,0,5,4,3,4,5,0,0,0,2,...,0,0,0,0,0,0,0,0,0,0
User_004,0,5,0,2,0,3,0,0,0,0,...,0,0,0,0,3,5,0,0,0,0
User_005,0,0,0,0,0,0,0,0,0,3,...,0,0,0,0,0,0,0,0,3,0


In [5]:
df.shape

(100, 30)

In [6]:
mem = df.memory_usage(deep = True)
print(mem)

Index                    5700
The_Dark_Knight           800
Inception                 800
Interstellar              800
The_Matrix                800
Avengers_Endgame          800
Parasite                  800
Joker                     800
1917                      800
Ford_v_Ferrari            800
Once_Upon_Hollywood       800
SpiderMan_NWH             800
Dune                      800
The_Batman                800
Top_Gun_Maverick          800
Everything_Everywhere     800
Avatar_WoW                800
Black_Panther             800
Doctor_Strange            800
Thor_LandT                800
Oppenheimer               800
Barbie                    800
Killers_Flower_Moon       800
Past_Lives                800
Saltburn                  800
Poor_Things               800
Zone_of_Interest          800
Maestro                   800
Priscilla                 800
Napoleon                  800
May_December              800
dtype: int64


In [7]:
print(f"Total Memory: {mem.sum():,}, bytes  ({mem.sum()/1024:.2f} Kb)")

Total Memory: 29,700, bytes  (29.00 Kb)


In [8]:
total = df.shape[0] * df.shape[1]
zeros = (df==0).sum().sum()
nonZero = total - zeros
print(f"Total Cells: {total:,}")
print(f"Zero Cells:  {zeros:,}  ({zeros/total*100:.1F}%) <- wasted")
print(f"Non Zero Cells:  {nonZero:,}  ({nonZero/total*100:.1F}%) <- Real Data")

Total Cells: 3,000
Zero Cells:  2,477  (82.6%) <- wasted
Non Zero Cells:  523  (17.4%) <- Real Data


In [9]:
sparse = csr_matrix(df.values)
sparse_bytes = (sparse.data.nbytes+sparse.indices.nbytes+sparse.indptr.nbytes)
print(sparse)
print(f"SParse Memory:  {sparse_bytes:,} bytes {sparse_bytes/1024:,.2f}")
print(f"Memory Saved:  {(mem.sum() - sparse_bytes)/1024:,.2f} Kb")

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 523 stored elements and shape (100, 30)>
  Coords	Values
  (0, 8)	5
  (0, 9)	3
  (0, 15)	3
  (0, 17)	4
  (0, 23)	4
  (0, 24)	4
  (0, 27)	2
  (0, 29)	4
  (1, 15)	5
  (1, 19)	5
  (1, 26)	5
  (1, 29)	5
  (2, 1)	5
  (2, 2)	4
  (2, 3)	3
  (2, 4)	4
  (2, 5)	5
  (2, 9)	2
  (2, 10)	2
  (3, 1)	5
  (3, 3)	2
  (3, 5)	3
  (3, 15)	3
  (3, 17)	3
  (3, 18)	4
  :	:
  (95, 28)	5
  (96, 13)	5
  (96, 17)	3
  (96, 18)	4
  (96, 19)	4
  (97, 1)	4
  (97, 6)	3
  (97, 7)	2
  (97, 13)	3
  (97, 16)	5
  (97, 22)	3
  (97, 29)	5
  (98, 4)	5
  (98, 8)	3
  (98, 9)	4
  (98, 16)	1
  (98, 17)	5
  (98, 19)	5
  (98, 25)	4
  (99, 7)	5
  (99, 12)	5
  (99, 15)	4
  (99, 18)	5
  (99, 22)	4
  (99, 26)	4
SParse Memory:  6,680 bytes 6.52
Memory Saved:  22.48 Kb


In [10]:
df_random = pd.DataFrame(np.random.randint(0,100, size = (10000000, 10)))
df_random.to_csv("data_random.csv")

In [11]:
%%time
df = pd.read_csv("data_random.csv")

CPU times: total: 5.2 s
Wall time: 5.37 s


In [12]:
!pip install pyarrow


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
%%time
df.to_csv("data_random.csv")

CPU times: total: 22.5 s
Wall time: 23.4 s


In [14]:
%%time
df.to_parquet("data_random.parquet")

CPU times: total: 2.5 s
Wall time: 2.6 s


In [15]:
%%time
df.to_feather("data_random.feather")

CPU times: total: 2 s
Wall time: 674 ms


In [16]:
%%time
df_parq = pd.read_parquet("data_random.parquet")

CPU times: total: 2.25 s
Wall time: 556 ms


In [17]:
%%time
df_feather = pd.read_feather("data_random.feather")

CPU times: total: 2.16 s
Wall time: 752 ms


In [18]:
%%time
df_csv = pd.read_csv("data_random.csv")

CPU times: total: 7.33 s
Wall time: 7.38 s


In [20]:
git in

SyntaxError: invalid syntax (2830201818.py, line 1)